## Preparacion para Lectura, RAG y Agente

### 1. Instalación base
Estas celdas instalan las dependencias principales del notebook: LangChain, OpenAI, loaders de PDF y utilidades de text splitting.

In [ ]:
pip install --upgrade pip

In [ ]:
%pip install -q langchain langchain-openai langchain-community pypdf langchain-text-splitters

In [ ]:
%pip install -q -U langchain-core

# Instalamos NLTK (Natural Language Toolkit), una librería de NLP en Python.

En tu notebook se usa en la celda siguiente para bajar recursos de tokenización y tagging:

### 2. Dependencias para embeddings y experimentación
Este bloque instala PyTorch, Transformers y extensiones experimentales de LangChain que vamos a usar más adelante para pruebas y comparación de estrategias.

In [ ]:
%pip install -q nltk

In [ ]:
%pip install -q "torch==2.2.2" "transformers<5"

In [ ]:
%pip install -q -U langchain-experimental

### 3. Verificación del entorno
Estas celdas validan que las versiones instaladas realmente estén disponibles en el kernel activo y descargan recursos auxiliares de NLTK.

In [ ]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

In [ ]:

import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

#Preparamos para importar variables de entorno

### 4. Variables de entorno y credenciales
Este bloque carga la configuración local y, si falta la API key, la pide de forma interactiva para no dejarla hardcodeada en el notebook.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
import getpass

if not openai_api_key:
    openai_api_key = getpass.getpass()

### 5. Carga del documento fuente
Acá se abre el PDF de prueba, se extrae el texto y se hace una primera validación para saber si el archivo trae texto embebido o si más adelante conviene OCR.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_para_ocr_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf"
pdf_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104046.pdf"

docs = PyPDFLoader(pdf_para_ocr_path).load()
pdf_text = "\n\n".join(doc.page_content for doc in docs).strip()

print(f"Paginas cargadas: {len(docs)}")
print(f"Chars extraidos: {len(pdf_text)}")

if not pdf_text:
    print("Este PDF parece escaneado o sin texto embebido. Usa OCR con el endpoint de ingesta y pdf_para_ocr_path.")
    

In [ ]:
pdf_text

In [ ]:
%pip install -q "torch==2.2.2" "transformers<5"

## Modelo de Embeddings y Tokenizador
- Vamos a Usar Transformers

### 6. Modelo local de embeddings para laboratorio
Estas celdas cargan un modelo local solo para experimentar y comparar. No es la ruta principal de producción si seguimos con OpenAI.

In [ ]:
import torch
AutoTokenizer = transformers.AutoTokenizer
AutoModel = transformers.AutoModel

print("torch:", torch.__version__)

emb_tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
emb_model = AutoModel.from_pretrained("intfloat/multilingual-e5-small")
print("Modelo cargado OK")

### 7. Splitter semántico con OpenAI
Acá se crea el splitter semántico basado en embeddings de OpenAI. Todavía no define el chunk final de producción; por ahora nos sirve para experimentar después de limpiar el texto legal.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker  # type: ignore[reportMissingImports]
from langchain_openai.embeddings import OpenAIEmbeddings

semantic_openai_splitter = SemanticChunker(OpenAIEmbeddings(api_key=openai_api_key))

### Módulo A — Leyes Sancionadas (Infoleg)
**Fuente de Verdad Firme.** Si el documento es una sanción, se carga directamente desde Infoleg (`texact.htm`). El texto ya es limpio y estructurado: **no se corre OCR, no se limpia nada**.

Flujo:
- `tipo_documento = "sancion"` → carga desde Infoleg, saltea la Sección 8
- `tipo_documento = "proyecto"` → continúa con el pipeline OCR de la Sección 8

In [ ]:
%pip install -q beautifulsoup4 requests

In [ ]:
import re
import hashlib
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# ─────────────────────────────────────────────────────────────────────────────
# URLs base de Infoleg
# ─────────────────────────────────────────────────────────────────────────────
_INFOLEG_BASE = "https://servicios.infoleg.gob.ar/infolegInternet"
_INFOLEG_ATTACH_BASE = f"{_INFOLEG_BASE}/anexos"


def _buscar_id_norma_infoleg(numero_ley: int, tipo_norma: int = 1, anio_sancion: str = "") -> tuple[int, str]:
    """
    Paso 1: buscar norma por tipo+numero en buscarNormas.do y extraer el id
    del primer <a href="/infolegInternet/verNorma.do?id=XXXXX"> del HTML.
    tipo_norma=1 corresponde a LEY.
    """
    search_url = (
        f"{_INFOLEG_BASE}/buscarNormas.do?tipoNorma={tipo_norma}"
        f"&numero={numero_ley}&anioSancion={anio_sancion}"
    )

    resp = requests.get(search_url, timeout=20)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding or "utf-8"

    # Extrae el id del primer <a href="/infolegInternet/verNorma.do?id=..."> del HTML
    m = re.search(
        r'<a href="/infolegInternet/verNorma\.do(?:;[^?]*)?\?id=(\d+)"',
        resp.text,
        re.IGNORECASE,
    )
    if not m:
        raise ValueError(
            f"No se pudo extraer idNorma para la ley {numero_ley} desde buscarNormas.do"
        )

    return int(m.group(1)), search_url


def _url_ver_norma(infoleg_id: int) -> str:
    return f"{_INFOLEG_BASE}/verNorma.do?id={infoleg_id}"


def _url_texact_fallback(infoleg_id: int) -> str:
    """Fallback clásico por rango de anexos cuando no aparece link directo en verNorma."""
    low = (infoleg_id // 10000) * 10000
    high = low + 9999
    return f"{_INFOLEG_ATTACH_BASE}/{low}-{high}/{infoleg_id}/texact.htm"


def _url_norma_fallback(infoleg_id: int) -> str:
    """URL del texto original de la norma (norma.htm)."""
    low = (infoleg_id // 10000) * 10000
    high = low + 9999
    return f"{_INFOLEG_ATTACH_BASE}/{low}-{high}/{infoleg_id}/norma.htm"


def _url_vinculos(infoleg_id: int, modo: int) -> str:
    """modo=1: actualizaciones que hace la norma; modo=2: normas que la actualizan."""
    return f"{_INFOLEG_BASE}/verVinculos.do?modo={modo}&id={infoleg_id}"


def _extraer_url_texact_desde_ver_norma(ver_norma_html: str, ver_norma_url: str) -> str | None:
    """Intenta encontrar un href a texact.htm en la página de verNorma."""
    soup = BeautifulSoup(ver_norma_html, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "texact.htm" in href.lower():
            return urljoin(ver_norma_url, href)

    return None


def _parse_infoleg_html(
    soup: BeautifulSoup,
    numero_ley: int,
    infoleg_id: int,
    source_url: str,
    sha256_hash: str,
) -> list[dict]:
    """Segmenta el HTML de Infoleg en unidades por artículo."""
    article_re = re.compile(r"^\s*(art[íi]culo|art\.)\s*[\dA-Za-z°º]+", re.IGNORECASE)
    heading_re = re.compile(r"^\s*(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", re.IGNORECASE)

    base_meta = {
        "source": "INFOLEG",
        "status": "VIGENTE",
        "law_id": numero_ley,
        "infoleg_id": infoleg_id,
        "source_url": source_url,
        "sha256_hash": sha256_hash,
    }

    units: list[dict] = []
    current_kind = "preambulo"
    current_title = "PREAMBULO"
    buffer: list[str] = []
    heading_stack: list[str] = []

    def flush() -> None:
        nonlocal buffer, current_kind, current_title
        text = " ".join(buffer).strip()
        if text:
            units.append(
                {
                    "kind": current_kind,
                    "title": current_title,
                    "path": " > ".join(heading_stack),
                    "text": text,
                    **base_meta,
                }
            )
        buffer = []

    for elem in soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "b"]):
        text = elem.get_text(separator=" ", strip=True)
        if not text or len(text) < 3:
            continue

        if article_re.match(text):
            flush()
            current_kind = "articulo"
            current_title = text[:160]
            buffer.append(text)
            continue

        m = heading_re.match(text)
        if m:
            flush()
            current_kind = m.group(1).lower()
            current_title = text[:160]
            heading_stack.append(current_title)
            buffer.append(text)
            continue

        buffer.append(text)

    flush()
    return units


def cargar_ley_infoleg(numero_ley: int) -> tuple[list[dict], dict]:
    """
    Carga una ley sancionada desde Infoleg siguiendo el flujo oficial:
    1) buscarNormas.do -> 2) verNorma.do?id=... -> 3) texact.htm
    """
    infoleg_id, search_url = _buscar_id_norma_infoleg(numero_ley)
    ver_norma_url = _url_ver_norma(infoleg_id)

    ver_resp = requests.get(ver_norma_url, timeout=20)
    ver_resp.raise_for_status()
    ver_resp.encoding = ver_resp.apparent_encoding or "utf-8"

    url_texact = _extraer_url_texact_desde_ver_norma(ver_resp.text, ver_norma_url)
    if not url_texact:
        url_texact = _url_texact_fallback(infoleg_id)

    # Mantiene exactamente el mismo árbol de anexos que el texto actualizado
    if "texact.htm" in url_texact.lower():
        url_norma = re.sub(r"texact\.htm(?:\?.*)?$", "norma.htm", url_texact, flags=re.IGNORECASE)
    else:
        url_norma = _url_norma_fallback(infoleg_id)

    texact_resp = requests.get(url_texact, timeout=30)
    texact_resp.raise_for_status()
    texact_resp.encoding = texact_resp.apparent_encoding or "utf-8"

    sha256_hash = hashlib.sha256(texact_resp.content).hexdigest()
    soup = BeautifulSoup(texact_resp.text, "html.parser")

    article_units = _parse_infoleg_html(
        soup,
        numero_ley=numero_ley,
        infoleg_id=infoleg_id,
        source_url=url_texact,
        sha256_hash=sha256_hash,
    )

    title_text = soup.title.get_text(strip=True) if soup.title else f"Ley {numero_ley}"

    metadata_doc = {
        "source": "INFOLEG",
        "status": "VIGENTE",
        "law_id": numero_ley,
        "infoleg_id": infoleg_id,
        "source_norma": url_norma,
        "source_actualizado": url_texact,
        "source_url": url_texact,
        "search_url": search_url,
        "ver_norma_url": ver_norma_url,
        "vinculos_actualiza_url": _url_vinculos(infoleg_id, modo=1),
        "vinculos_actualizado_por_url": _url_vinculos(infoleg_id, modo=2),
        "sha256_hash": sha256_hash,
        "titulo": title_text,
    }

    return article_units, metadata_doc


print("Módulo A (Infoleg) cargado OK con flujo buscarNormas -> verNorma -> texact.")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ROUTING — cambiar estos dos valores según el documento a procesar
# ─────────────────────────────────────────────────────────────────────────────
tipo_documento = "sancion"   # "sancion" | "proyecto"
numero_ley     = 20744       # solo se usa si tipo_documento == "sancion"


def _attach_doc_metadata_to_units(units: list[dict], metadata: dict, keys: list[str]) -> list[dict]:
    """Asegura que metadatos de documento estén presentes en cada unidad/chunk."""
    out: list[dict] = []
    for u in units:
        enriched = dict(u)
        for k in keys:
            if k in metadata and metadata.get(k) is not None:
                enriched[k] = metadata.get(k)
        out.append(enriched)
    return out


# ─────────────────────────────────────────────────────────────────────────────
if tipo_documento == "sancion":
    article_units, doc_metadata = cargar_ley_infoleg(numero_ley)

    # Metadatos críticos que deben viajar en cada chunk
    article_units = _attach_doc_metadata_to_units(
        article_units,
        doc_metadata,
        keys=[
            "source",
            "status",
            "law_id",
            "infoleg_id",
            "titulo",
            "search_url",
            "ver_norma_url",
            "vinculos_actualiza_url",
            "vinculos_actualizado_por_url",
            "source_norma",
            "source_actualizado",
            "source_url",
            "sha256_hash",
        ],
    )

    print(f"Ley {numero_ley} cargada desde Infoleg.")
    print(f"Título:        {doc_metadata.get('titulo', '')}")
    print(f"Buscar URL:    {doc_metadata.get('search_url', '')}")
    print(f"Ver norma URL: {doc_metadata.get('ver_norma_url', '')}")
    print(f"Vínculos (modo=1, actualiza):       {doc_metadata.get('vinculos_actualiza_url', '')}")
    print(f"Vínculos (modo=2, la actualizan):   {doc_metadata.get('vinculos_actualizado_por_url', '')}")
    print(f"Source norma:       {doc_metadata.get('source_norma', '')}")
    print(f"Source actualizado: {doc_metadata.get('source_actualizado', '')}")
    print(f"SHA-256:       {doc_metadata.get('sha256_hash', '')}")
    print(f"Unidades:      {len(article_units)}")
    print(f"Artículos:     {sum(1 for u in article_units if u['kind'] == 'articulo')}")
    print("\n→ Saltear la Sección 8 (OCR pipeline). No es necesaria.")

else:
    # Metadata base para flujo OCR (también indispensable para chunks)
    doc_metadata = {
        "source": "BORA_OCR",
        "status": "PROYECTO",
        "law_id": None,
        "infoleg_id": None,
        "titulo": "Proyecto (OCR)",
        "search_url": None,
        "ver_norma_url": None,
        "vinculos_actualiza_url": None,
        "vinculos_actualizado_por_url": None,
        "source_norma": None,
        "source_actualizado": pdf_para_ocr_path,
        "source_url": pdf_para_ocr_path,
        "sha256_hash": None,
    }
    print("Documento es Proyecto (BORA/OCR) → continuar con la Sección 8.")


# Usamos un LLM para leer la ley desde Infoleg
Este paso nos va ahorrar tiempo y evitar problemas de parseo.

In [ ]:
Base_System_Prompt = "Vas a actuar como un Asesor Legislativo Experto y sin ideologias politicas en interpretar leyes argentinas que se acaban de publicar en el boletin oficial de la republica argentina y estan disponibles en InfoLEG. Simplemente extraer su texto AS-IS para luego poder usarlo para hacer un chunking para guarlo en una DB de RAG. Siempre debes basar tus respuestas exclusivamente en el contenido de la ley y evitar cualquier sesgo o interpretación personal."

In [ ]:
System_Prompt_Consultas = "Vas a actuar como un Asesor Legislativo Experto y sin ideologias politicas en interpretar leyes argentinas que se acaban de publicar en el boletin oficial de la republica argentina y estan disponibles en InfoLEG. Tu tarea es ayudar a entender el contenido de estas leyes, explicando su articulado, contexto y posibles implicancias de manera clara y precisa. No debes inventar información ni hacer suposiciones sin base en el texto legal. Si no sabes la respuesta o el texto no lo especifica, debes decir que no se puede determinar a partir del texto. Siempre debes basar tus respuestas exclusivamente en el contenido de la ley y evitar cualquier sesgo o interpretación personal."



In [ ]:
import requests
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI

# Fuente de verdad: texto directo de Infoleg (sin pasar por LLM para evitar refusals)
url = "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm"

html = requests.get(url, timeout=30)
html.raise_for_status()
html.encoding = html.apparent_encoding or "utf-8"

# Texto plano completo de la ley
texto_fuente_directo = BeautifulSoup(html.text, "html.parser").get_text(" ", strip=True)

# Compatibilidad hacia atrás con celdas existentes que esperan 'respuesta'
respuesta = texto_fuente_directo

# Modelo para las etapas de QA/RAG (no para extracción del texto legal)
modelo = ChatOpenAI(model="gpt-4o-mini")

print(f"Texto legal cargado desde Infoleg (chars): {len(texto_fuente_directo)}")
print("OK: extracción directa, sin intervención de LLM.")

## Splitter Semantico
- El texto_fuente_directo se cargo desde el sitio web usando un LLM  y BeautifulSoup

In [ ]:
import re
from langchain_core.documents import Document

# 1) Selección robusta de fuente de texto (prioriza extracción directa de Infoleg)
if "texto_fuente_directo" in globals() and isinstance(texto_fuente_directo, str) and texto_fuente_directo.strip():
    texto_fuente = texto_fuente_directo
elif "texto" in globals() and isinstance(texto, str) and texto.strip():
    texto_fuente = texto
elif isinstance(respuesta, str):
    texto_fuente = respuesta
elif hasattr(respuesta, "text") and isinstance(respuesta.text, str):
    texto_fuente = respuesta.text
else:
    texto_fuente = str(respuesta)

texto_fuente = re.sub(r"\s+", " ", texto_fuente).strip()
if not texto_fuente:
    raise ValueError("No hay texto fuente para chunking. Ejecuta primero la celda de carga desde Infoleg.")

# 2) Splitter semántico legal por encabezados y artículos
heading_start_re = re.compile(
    r"\*{0,2}\s*(?:T[ÍI]TULO|CAP[ÍI]TULO|SECCI[ÓO]N|LIBRO|ANEXO)\s+[IVXLCDM0-9]+"
)

# Solo considera encabezado de artículo cuando hay patrón de inicio real con guion:
# Art. 2° — ...   o   Artículo 10. — ...
article_header_pattern = (
    r"\*{0,2}\s*(?:Art\.|Artículo)\s*\d+[a-zA-Zº°]*\.?\s*"
    r"(?:bis|ter|quater|quinquies|sexies|septies|octies|nonies)?\s*[—-]\s*"
)
article_start_re = re.compile(article_header_pattern, re.IGNORECASE)

marker_re = re.compile(
    r"\*{0,2}\s*(?:T[ÍI]TULO|CAP[ÍI]TULO|SECCI[ÓO]N|LIBRO|ANEXO)\s+[IVXLCDM0-9]+|" + article_header_pattern,
    re.IGNORECASE,
)

def _heading_level(title: str) -> int:
    t = title.upper()
    if re.search(r'\bLIBRO\b', t):
        return 0
    if re.search(r'\bANEXO\b', t):
        return 0
    if re.search(r'T[ÍI]TULO', t):
        return 1
    if re.search(r'CAP[ÍI]TULO', t):
        return 2
    if re.search(r'SECCI[ÓO]N', t):
        return 3
    return 99

def _get_capitulo(stack: list[str]) -> str | None:
    for entry in reversed(stack):
        if re.search(r'CAP[ÍI]TULO', entry.upper()):
            return entry
    return None

def _parse_articulo_ref(texto_art: str) -> tuple[int | None, str | None, str | None]:
    m = re.search(
        r"\b(?:Art\.|Artículo)\s*(\d+)\s*([a-zA-Z]+)?",
        texto_art,
        flags=re.IGNORECASE,
    )
    if not m:
        return None, None, None
    nro = int(m.group(1))
    suf = m.group(2).lower() if m.group(2) else None
    if suf and suf not in {"bis", "ter", "quater", "quinquies", "sexies", "septies", "octies", "nonies"}:
        suf = None
    ref = f"{nro} {suf}".strip() if suf else str(nro)
    return nro, suf, ref

def _es_nota_substitucion_texto(texto: str) -> bool:
    b = texto.lower()
    patrones = [
        "nota infoleg",
        "sustituido por art.",
        "sustituida por art.",
        "incorporado por art.",
        "incorporada por art.",
        "modificado por art.",
        "modificada por art.",
        "texto segun",
        "vigencia:",
    ]
    return any(p in b for p in patrones)

def _es_nota_substitucion(block: str) -> bool:
    return _es_nota_substitucion_texto(block) and len(block) < 1500

def _extraer_notas_finales(block: str) -> tuple[str, list[str]]:
    """Saca notas parentéticas finales del cuerpo del artículo y las devuelve aparte."""
    notas: list[str] = []
    texto = block.strip()

    while True:
        match = re.search(r"\s*(\([^()]{20,1200}\))\s*$", texto, flags=re.IGNORECASE)
        if not match:
            break
        candidata = match.group(1)
        if not _es_nota_substitucion_texto(candidata):
            break
        notas.insert(0, candidata)
        texto = texto[:match.start()].rstrip()

    return texto, notas

def split_ley_semantico_articulos(texto: str) -> tuple[list[dict], dict[str, list[str]]]:
    markers = list(marker_re.finditer(texto))
    if not markers:
        return ([{"kind": "texto", "title": "TEXTO", "text": texto, "path": "", "articulo_nro": None, "articulo_suffix": None, "articulo_ref": None, "capitulo": None}], {})

    chunks = []
    notes_by_ref: dict[str, list[str]] = {}
    heading_stack: list[str] = []

    if markers[0].start() > 0:
        pre = texto[:markers[0].start()].strip()
        if pre:
            chunks.append({
                "kind": "preambulo",
                "title": "PREAMBULO",
                "text": pre,
                "path": "",
                "articulo_nro": None,
                "articulo_suffix": None,
                "articulo_ref": None,
                "capitulo": None,
            })

    for index, marker in enumerate(markers):
        start = marker.start()
        end = markers[index + 1].start() if index + 1 < len(markers) else len(texto)
        block = texto[start:end].strip()
        if not block:
            continue

        if heading_start_re.match(block):
            kind = "heading"
            title = re.split(
                r"(?=" + article_header_pattern + r")",
                block,
                maxsplit=1,
                flags=re.IGNORECASE,
            )[0].strip()

            new_level = _heading_level(title)
            while heading_stack and _heading_level(heading_stack[-1]) >= new_level:
                heading_stack.pop()
            heading_stack.append(title)
            path = " > ".join(heading_stack)

            chunks.append({
                "kind": kind,
                "title": title,
                "text": block,
                "path": path,
                "articulo_nro": None,
                "articulo_suffix": None,
                "articulo_ref": None,
                "capitulo": _get_capitulo(heading_stack),
            })

        elif article_start_re.match(block):
            articulo_nro, articulo_suffix, articulo_ref = _parse_articulo_ref(marker.group())

            # Ignorar notas de sustitución/incorporación como chunks y guardarlas como metadata
            if _es_nota_substitucion(block):
                ref = articulo_ref or "sin_ref"
                notes_by_ref.setdefault(ref, []).append(block)
                continue

            # Mover notas parentéticas finales del cuerpo a metadata
            block_limpio, notas_finales = _extraer_notas_finales(block)
            if notas_finales:
                ref = articulo_ref or "sin_ref"
                notes_by_ref.setdefault(ref, []).extend(notas_finales)
            block = block_limpio

            kind = "articulo"
            match_title = re.match(
                r"(?s)(" + article_header_pattern + r"[^*\n]{0,180})",
                block,
                flags=re.IGNORECASE,
            )
            title = match_title.group(1).strip() if match_title else block[:180]
            path = " > ".join(heading_stack)

            chunks.append({
                "kind": kind,
                "title": title,
                "text": block,
                "path": path,
                "articulo_nro": articulo_nro,
                "articulo_suffix": articulo_suffix,
                "articulo_ref": articulo_ref,
                "capitulo": _get_capitulo(heading_stack),
            })

        else:
            chunks.append({
                "kind": "texto",
                "title": block[:180],
                "text": block,
                "path": " > ".join(heading_stack),
                "articulo_nro": None,
                "articulo_suffix": None,
                "articulo_ref": None,
                "capitulo": _get_capitulo(heading_stack),
            })

    return chunks, notes_by_ref

chunks_estructurados, notes_by_ref = split_ley_semantico_articulos(texto_fuente)
chunks_ley = [chunk["text"] for chunk in chunks_estructurados]

doc_meta_min = {
    "source": doc_metadata.get("source") if isinstance(doc_metadata, dict) else "INFOLEG",
    "status": doc_metadata.get("status") if isinstance(doc_metadata, dict) else "VIGENTE",
    "law_id": doc_metadata.get("law_id") if isinstance(doc_metadata, dict) else None,
    "infoleg_id": doc_metadata.get("infoleg_id") if isinstance(doc_metadata, dict) else None,
    "titulo": doc_metadata.get("titulo") if isinstance(doc_metadata, dict) else None,
    "search_url": doc_metadata.get("search_url") if isinstance(doc_metadata, dict) else None,
    "ver_norma_url": doc_metadata.get("ver_norma_url") if isinstance(doc_metadata, dict) else None,
    "vinculos_actualiza_url": doc_metadata.get("vinculos_actualiza_url") if isinstance(doc_metadata, dict) else None,
    "vinculos_actualizado_por_url": doc_metadata.get("vinculos_actualizado_por_url") if isinstance(doc_metadata, dict) else None,
    "source_norma": doc_metadata.get("source_norma") if isinstance(doc_metadata, dict) else None,
    "source_actualizado": doc_metadata.get("source_actualizado") if isinstance(doc_metadata, dict) else None,
    "source_url": doc_metadata.get("source_url") if isinstance(doc_metadata, dict) else None,
    "sha256_hash": doc_metadata.get("sha256_hash") if isinstance(doc_metadata, dict) else None,
}

docs_ley_chunks = []
for index, chunk in enumerate(chunks_estructurados, start=1):
    ref = chunk.get("articulo_ref")
    notas_substitucion = notes_by_ref.get(ref, []) if ref else []
    docs_ley_chunks.append(
        Document(
            page_content=chunk["text"],
            metadata={
                **doc_meta_min,
                "chunk_index": index,
                "chunk_count": len(chunks_estructurados),
                "kind": chunk["kind"],
                "title": chunk["title"],
                "path": chunk["path"],
                "articulo_nro": chunk["articulo_nro"],
                "articulo_suffix": chunk["articulo_suffix"],
                "articulo_ref": chunk["articulo_ref"],
                "capitulo": chunk["capitulo"],
                "notas_substitucion": notas_substitucion,
                "cantidad_notas_substitucion": len(notas_substitucion),
            },
        )
    )

print(f"Fuente usada para chunking: {'texto_fuente_directo' if 'texto_fuente_directo' in globals() else 'fallback'}")
print(f"Chunks semánticos generados: {len(chunks_estructurados)}")
print(f"  heading : {sum(1 for c in chunks_estructurados if c['kind'] == 'heading')}")
print(f"  articulo: {sum(1 for c in chunks_estructurados if c['kind'] == 'articulo')}")
print(f"Notas de sustitución ignoradas como chunk: {sum(len(v) for v in notes_by_ref.values())}")

print("\nSample de 5 artículos con nro/ref/capítulo:")
arts = [c for c in chunks_estructurados if c["kind"] == "articulo"]
for c in arts[:5]:
    print(f"  Art. {c['articulo_ref']} | capitulo: {str(c['capitulo'])[:60]}")

art_2_doc = next((d for d in docs_ley_chunks if d.metadata.get("articulo_ref") == "2" and d.metadata.get("kind") == "articulo"), None)
if art_2_doc:
    print("\nArt. 2 -> notas_substitucion:", art_2_doc.metadata.get("cantidad_notas_substitucion"))


In [ ]:
chunks_ley

## Preparando FAISS
Vamos a usar FAISS como DB Vectorial por la velocidad que nos proporciona para la busqueda

In [ ]:
%pip install faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(documents=docs_ley_chunks, embedding=embeddings)

retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 20, "lambda_mult": 0.6},
)

print(f"Chunks indexados en FAISS: {len(docs_ley_chunks)}")
print("Metadata del primer documento indexado:")
for key in ["law_id", "infoleg_id", "titulo", "kind", "title", "path", "source_actualizado"]:
    print(f"  {key}: {docs_ley_chunks[0].metadata.get(key)}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template_question = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Responde SOLO con el contenido del contexto.

Reglas obligatorias:
1) Si el usuario pregunta por un artículo específico (ej. "artículo 245") y ese artículo no está en el contexto recuperado, responde exactamente:
   "El artículo {{N}} no se encuentra en el corpus disponible de esta ley (texto consolidado cargado). No se puede determinar su contenido literal con la fuente actual."
2) Si hay referencias indirectas al artículo faltante en otros artículos (ej. "según el artículo N"), puedes mencionarlas como contexto secundario, aclarando que NO es el texto del artículo solicitado.
3) No inventes contenido ni completes por conocimiento externo.
4) Si el texto no alcanza, indícalo de forma explícita y breve.

Contexto:\n{contexto}""",
        ),
        ("human", "{query}"),
    ]
)

pipeline = prompt_template_question | modelo | StrOutputParser()


- reescritura de la pregunta

In [ ]:
pregunta = (
    "Es falso que se habiliten jornadas arbitrarias de 12 horas?"  # Definindo a pergunta
)

In [ ]:
from langchain_core.prompts import PromptTemplate

query_model = ChatOpenAI(model="gpt-4.1-nano")

rewriter_prompt_template = PromptTemplate.from_template(
    """
Genera preguntas de investigación para la base vectorial a partir de la pregunta del usuario, permitiendo una respuesta más precisa por medio de la búsqueda semántica.
Alcanza con retornar la consulta revisada de la DB vectorial, entre comillas.

Pregunta del usuario: {pregunta}

Consulta revisada del Vector DB:
"""
)

rewriter_pipeline = rewriter_prompt_template | query_model | StrOutputParser()
pregunta_reescrita = rewriter_pipeline.invoke(pregunta)
pregunta_reescrita

In [ ]:

pregunta

In [ ]:
# Usa SIEMPRE el flujo híbrido para recuperar contexto
debug_out = responder_con_rag(pregunta_usuario=pregunta, k=8, usar_reescritura=True)
trechos = debug_out["docs_recuperados"]
contexto = debug_out["contexto"]


In [ ]:
print("Artículos mencionados:", debug_out["articulos_mencionados"])
print("Artículos encontrados:", debug_out["articulos_encontrados"])
print("Artículos faltantes:", debug_out["articulos_faltantes"])
print("\nRespuesta:")
print(debug_out["respuesta"])
print("\nChunks recuperados:")
trechos


In [ ]:
hyde_prompt_template = """
Responda a pergunta da melhor forma possivel em no minimo 2 paragrafos únicamente utilizando el contenido proporcionado como única fuente de verdad. \n\nContexto:\n{contexto}. Si el texto no especifica la respuesta, di que no se puede determinar a partir del texto. Evita cualquier interpretación o suposición sin base en el texto.
Pergunta: {pregunta}
"""

hyde_prompt = PromptTemplate.from_template(hyde_prompt_template)

In [ ]:
hyde_pipeline = hyde_prompt | query_model | StrOutputParser()
hyde_pipeline.invoke({"pregunta": pregunta, "contexto": contexto})

# Motor de Busqueda IA

### Prerrequisitos del bloque Funciones
Estas funciones asumen que el notebook ya dejó preparado el corpus legal y el pipeline RAG. Antes de usarlas, ejecutá estas etapas en orden:

1. **Cargar credenciales y modelos**
   - `openai_api_key` disponible.
   - `modelo` inicializado para respuesta.
   - `query_model` inicializado si vas a usar reescritura de consultas.

2. **Leer la fuente legal**
   - Para leyes vigentes desde Infoleg: ejecutar la carga que deja `doc_metadata` y `texto_fuente_directo`.
   - Para documentos OCR/proyecto: ejecutar la limpieza que deja `pdf_text_clean` y `article_units`, junto con `doc_metadata`.

3. **Hacer el chunking legal**
   - Ejecutar la celda que construye `chunks_estructurados`, `notes_by_ref`, `chunks_ley` y `docs_ley_chunks`.
   - En este punto cada chunk ya debe tener metadata jurídica (`articulo_nro`, `articulo_ref`, `capitulo`, `path`) y metadata de origen (`source_url`, `sha256_hash`, etc.).

4. **Indexar el corpus en FAISS**
   - Ejecutar la celda que crea `vector_store` y `retriever` a partir de `docs_ley_chunks`.
   - Sin `retriever`, `responder_con_rag(...)` falla porque no tiene recuperación semántica disponible.

5. **Construir los prompts de QA**
   - Ejecutar la celda que crea `pipeline` para respuesta final.
   - Si querés reescritura automática de preguntas, ejecutar también la celda que crea `rewriter_pipeline`.

Estado mínimo esperado en memoria antes de llamar a `responder_con_rag(...)`: `doc_metadata`, `docs_ley_chunks`, `retriever`, `pipeline` y, opcionalmente, `rewriter_pipeline`. Si alguno de esos objetos no existe, hay que volver a correr las celdas de preparación anteriores.

In [ ]:
import re as _re

def _es_nota_complementaria(doc) -> bool:
    title = str(doc.metadata.get("title", "")).strip().lower()
    path = str(doc.metadata.get("path", "")).lower()

    # Las verdaderas notas suelen vivir en disposiciones complementarias o venir tituladas como acto de sustitución/incorporación
    if "disposiciones complementarias" in path:
        return True

    patrones_titulo = [
        "sustituido por art.",
        "sustituida por art.",
        "incorporado por art.",
        "incorporada por art.",
        "modificado por art.",
        "modificada por art.",
        "nota infoleg",
    ]
    return any(p in title for p in patrones_titulo)

def _score_doc_articulo(doc, nro: int, suf: str | None) -> int:
    score = 0
    doc_nro = doc.metadata.get("articulo_nro")
    doc_suf = doc.metadata.get("articulo_suffix")
    if doc_nro == nro:
        score += 10
    if (doc_suf or None) == (suf or None):
        score += 5
    if not _es_nota_complementaria(doc):
        score += 7
    # Prioriza encabezado textual exacto de artículo
    t = str(doc.metadata.get("title", "")).lower()
    ref = f"art. {nro}" if suf is None else f"art. {nro} {suf}"
    if ref in t or f"artículo {nro}" in t:
        score += 3
    return score

def _parse_articulos_pregunta(pregunta: str) -> list[tuple[int, str | None]]:
    # Captura: art. 245 / artículo 245 / artículo 245 bis
    out = []
    for m in _re.finditer(r'\b(?:art(?:ículo|iculo|\.)\s*)(\d+)\s*([a-zA-Z]+)?', pregunta, flags=_re.IGNORECASE):
        nro = int(m.group(1))
        suf = m.group(2).lower() if m.group(2) else None
        if suf and suf not in {"bis", "ter", "quater", "quinquies", "sexies", "septies", "octies", "nonies"}:
            suf = None
        out.append((nro, suf))
    # dedup preservando orden
    seen = set()
    unique = []
    for x in out:
        if x in seen:
            continue
        seen.add(x)
        unique.append(x)
    return unique

def _build_origen_documento() -> dict:
    if "doc_metadata" in globals() and isinstance(doc_metadata, dict):
        return {
            "source_norma": doc_metadata.get("source_norma"),
            "source_actualizado": doc_metadata.get("source_actualizado"),
            "source_url": doc_metadata.get("source_url"),
            "sha256_hash": doc_metadata.get("sha256_hash"),
        }

    fuente = None
    if "docs_ley_chunks" in globals() and docs_ley_chunks:
        fuente = docs_ley_chunks[0].metadata

    if fuente:
        return {
            "source_norma": fuente.get("source_norma"),
            "source_actualizado": fuente.get("source_actualizado"),
            "source_url": fuente.get("source_url"),
            "sha256_hash": fuente.get("sha256_hash"),
        }

    return {
        "source_norma": None,
        "source_actualizado": None,
        "source_url": None,
        "sha256_hash": None,
    }

def responder_con_rag(pregunta_usuario: str, k: int = 5, usar_reescritura: bool = True) -> dict:
    """
    Búsqueda híbrida (vectorial + reglas por artículo) robusta para casos como 245 vs 245 bis.
    """
    if not pregunta_usuario or not str(pregunta_usuario).strip():
        raise ValueError("La pregunta no puede estar vacía.")

    if "retriever" not in globals():
        raise RuntimeError("No existe 'retriever' en memoria. Ejecuta primero las celdas de FAISS/RAG.")
    if "pipeline" not in globals():
        raise RuntimeError("No existe 'pipeline' de respuesta. Ejecuta la celda del prompt RAG.")
    if "docs_ley_chunks" not in globals():
        raise RuntimeError("No existe 'docs_ley_chunks' en memoria. Ejecuta primero la celda de chunking.")

    pregunta_original = str(pregunta_usuario).strip()
    pregunta_busqueda = pregunta_original
    origen_documento = _build_origen_documento()

    if usar_reescritura and "rewriter_pipeline" in globals():
        try:
            pregunta_busqueda = rewriter_pipeline.invoke(pregunta_usuario).strip(' "\n\t')
        except Exception:
            pregunta_busqueda = pregunta_original

    # 1) Retrieval semántico
    trechos_local = retriever.invoke(pregunta_busqueda)
    trechos_local = trechos_local[:k] if isinstance(trechos_local, list) else []

    # 2) Parseo robusto de artículos pedidos
    articulos_pedidos = _parse_articulos_pregunta(pregunta_original)
    articulos_mencionados = [f"{n}{(' ' + s) if s else ''}" for n, s in articulos_pedidos]

    docs_articulos = [d for d in docs_ley_chunks if d.metadata.get("kind") == "articulo"]

    # 3) Match exacto nro+sufijo con scoring
    ids_ya = {t.id for t in trechos_local if hasattr(t, "id")}
    docs_encontrados = []
    articulos_encontrados = []
    articulos_faltantes = []

    for nro, suf in articulos_pedidos:
        candidatos = [d for d in docs_articulos if d.metadata.get("articulo_nro") == nro]
        if suf is not None:
            candidatos = [d for d in candidatos if (d.metadata.get("articulo_suffix") or None) == suf]
        # si pidió sin sufijo, prioriza sin sufijo (245 vs 245 bis)
        else:
            sin_suf = [d for d in candidatos if (d.metadata.get("articulo_suffix") or None) is None]
            if sin_suf:
                candidatos = sin_suf

        if not candidatos:
            articulos_faltantes.append(f"{nro}{(' ' + suf) if suf else ''}")
            continue

        best = sorted(candidatos, key=lambda d: _score_doc_articulo(d, nro, suf), reverse=True)[0]
        docs_encontrados.append(best)
        articulos_encontrados.append(f"{nro}{(' ' + suf) if suf else ''}")
        if best.id not in ids_ya:
            trechos_local.insert(0, best)
            ids_ya.add(best.id)

    # 4) Referencias indirectas solo para faltantes
    referencias_indirectas = []
    for art_ref in articulos_faltantes:
        nro = int(art_ref.split()[0])
        pat = _re.compile(rf'\bart[íi]culo\s+{nro}\b', flags=_re.IGNORECASE)
        for d in docs_articulos:
            if pat.search(d.page_content):
                if d.id not in ids_ya:
                    trechos_local.append(d)
                    ids_ya.add(d.id)
                referencias_indirectas.append(
                    {
                        "articulo_faltante": art_ref,
                        "encontrado_en_articulo": d.metadata.get("articulo_ref") or d.metadata.get("articulo_nro"),
                        "title": d.metadata.get("title", "")[:80],
                    }
                )

    contexto_local = "\n\n".join(t.page_content for t in trechos_local)

    # 5) Respuesta determinística para artículos explícitos
    if articulos_faltantes:
        faltan_txt = ", ".join(articulos_faltantes)
        respuesta_final = (
            f"El/los artículo(s) {faltan_txt} no se encuentra(n) en el corpus disponible de esta ley "
            "(texto consolidado cargado). No se puede determinar su contenido literal con la fuente actual."
        )
        if referencias_indirectas:
            refs = []
            usados = set()
            for r in referencias_indirectas:
                clave = (r["articulo_faltante"], r["encontrado_en_articulo"])
                if clave in usados:
                    continue
                usados.add(clave)
                refs.append(f"Art. {r['encontrado_en_articulo']} menciona al art. {r['articulo_faltante']}")
            if refs:
                respuesta_final += " Referencias indirectas encontradas: " + "; ".join(refs) + "."

    elif articulos_pedidos and docs_encontrados:
        partes = []
        for d in docs_encontrados:
            ref = d.metadata.get("articulo_ref") or d.metadata.get("articulo_nro")
            if _es_nota_complementaria(d):
                partes.append(f"Art. {ref}: referencia complementaria encontrada, no texto normativo principal.")
            elif "derogado" in str(d.metadata.get("title", "")).lower():
                partes.append(f"Art. {ref}: figura como derogado en el corpus cargado.")
            else:
                partes.append(f"Art. {ref}: {d.page_content}")
        respuesta_final = "\n\n".join(partes)

    else:
        respuesta_final = pipeline.invoke({
            "query": pregunta_original,
            "pregunta": pregunta_original,
            "contexto": contexto_local,
        })

    return {
        "pregunta_original": pregunta_original,
        "pregunta_busqueda": pregunta_busqueda,
        "origen_documento": origen_documento,
        "cantidad_chunks": len(trechos_local),
        "articulos_mencionados": articulos_mencionados,
        "articulos_encontrados": articulos_encontrados,
        "articulos_faltantes": articulos_faltantes,
        "referencias_indirectas": referencias_indirectas,
        "docs_recuperados": trechos_local,
        "chunks": [
            {
                "articulo_nro": t.metadata.get("articulo_nro"),
                "articulo_ref": t.metadata.get("articulo_ref"),
                "kind": t.metadata.get("kind"),
                "title": t.metadata.get("title", "")[:80],
            }
            for t in trechos_local
        ],
        "contexto": contexto_local,
        "respuesta": respuesta_final,
    }

print("Funcion 'responder_con_rag' lista con origen_documento separado en la salida JSON.")


In [ ]:
print(responder_con_rag(pregunta_usuario="¿La jornada laboral obligatoria pasa a ser de 12 horas?"))

In [ ]:
out = responder_con_rag("Que dice el articulo 245 de la ley")
print("origen_documento:")
print(out["origen_documento"])
print("\nrespuesta (preview 400):")
print(out["respuesta"][:400])


# Cargamos documentos por WebLoader

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader_noticias = WebBaseLoader(web_path=doc_metadata.get("ver_norma_url", ""))
ley_raw = loader_noticias.load()

In [ ]:
ley_raw

### 8. Limpieza y normalización legal
Este bloque remueve ruido de ingesta, protege encabezados jurídicos reales y deja el texto en unidades legales inspeccionables antes de pasar a chunking final.

### 8a. Diccionario legal y scoring de confianza OCR
Antes de descartar una línea "ambigua", le asignamos un **score de confianza** (0.0 = basura, 1.0 = confiable).
Esto nos permite:
- Descartar líneas severamente dañadas (score < 0.6)
- Revisar manualmente dudosas (0.6 ≤ score < 0.8)
- Aceptar líneas limpias (score ≥ 0.8)


In [ ]:
import re

# Diccionario básico de español + términos legales argentinos
LEGAL_SPANISH_DICT = {
    # Frecuentes
    "el", "la", "de", "que", "y", "a", "en", "se", "del", "con", "por", "para",
    "un", "una", "es", "este", "ese", "cual", "cuyo", "mas", "más", "no",
    "los", "las", "o", "cuando", "si", "donde", "como", "cuales",

    # Órganos e instrumentos
    "constitucion", "constitución", "nacional", "honorable",
    "camara", "cámara", "senadores", "diputados", "senado", "congreso", "parlamento",

    # Jurídico
    "articulo", "artículo", "inciso", "párrafo", "parágrafo", "apartado",
    "proyecto", "ley", "decreto", "resolucion", "resolución", "disposicion", "disposición",
    "expediente", "expedinte", "expedieente",  # variaciones OCR
    "presentado", "presentada", "presentados",
    "cuyo", "acto", "fecha", "firma", "firmante", "numerado",
    "aprobado", "sancionado", "sanción", "vigencia", "deroga", "derogacion", "abrogación",
    "abroga", "abrogaciones", "disposiciones", "precedentes", "contrarias",
    "derecho", "derechos", "obligacion", "obligaciones", "obligatoriedad",
    "persona", "personas", "ciudadano", "ciudadana", "argentino", "argentina",
    "empresa", "entidad", "institucion", "institución", "organismo", "organismos",
    "ministro", "ministra", "secretario", "secretaria", "secretaría",
    "articulos",

    # Verbos y conectores
    "segun", "según", "mediante", "establece", "deber", "consideracion", "consideración",
    "teniendo", "resulta", "resultan", "visto", "considerando", "entiende", "entienden",
    "durante", "dentro", "fuera", "frente", "ante",

    # Ordinales y numeración
    "primero", "primera", "segundo", "segunda", "tercero", "tercera",
    "numero", "número", "numeración",
}


def _is_protected_legal_line(line: str) -> bool:
    """True si la línea parece contener contenido legal válido que no se debe descartar."""
    legal_keywords = [
        "artículo", "articulo", "ley", "decreto", "resolución", "resolucion", "proyecto",
        "honorable", "cámara", "camara", "senado", "diputados", "congreso",
        "constitución", "constitucion", "nacional", "sancionado", "vigencia",
    ]
    line_lower = line.lower()
    return any(keyword in line_lower for keyword in legal_keywords)


def _ocr_confidence(line: str, context_lines: list[str] = None, legal_dict: set = None) -> float:
    """Calcula confianza de que la línea es contenido legítimo (0.0=basura, 1.0=confiable)."""
    if legal_dict is None:
        legal_dict = LEGAL_SPANISH_DICT

    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return 1.0

    score = 1.0

    # Penalización 1: ruido de símbolos frecuentes
    noise_chars = len(re.findall(r"[~_|{}\[\]`¬]", l))
    score -= min(0.3, noise_chars * 0.15)

    # Penalización 2: densidad de caracteres inusuales
    unusual = len(re.findall(r"[~_\\|{}\[\]`¬@#$%&*]", l))
    if l:
        unusual_ratio = unusual / len(l)
        if unusual_ratio > 0.2:
            score -= 0.4

    # Penalización 3: palabras raras
    words = re.findall(r"\b[a-záéíóúñ]+\b", l, re.IGNORECASE)
    words_lower = [w.lower() for w in words]

    if words:
        valid_words = sum(1 for w in words_lower if w in legal_dict or len(w) > 3)
        valid_ratio = valid_words / len(words)
        if valid_ratio < 0.5:
            score -= (1 - valid_ratio) * 0.3

    # Penalización 4: cambios de caso (garbled OCR)
    if len(l) < 80 and re.search(r"[a-z][A-Z][a-z]", l):
        score -= 0.25

    # Bonificación 1: vocabulario legal
    if words_lower and any(w in legal_dict for w in words_lower if len(w) > 3):
        score += 0.15

    # Bonificación 2: aparición en contexto cercano
    if context_lines and words_lower:
        context_text = " ".join(context_lines).lower()
        if any(w in context_text for w in words_lower if len(w) > 4):
            score += 0.1

    return max(0.0, min(1.0, score))


# Prueba rápida del scorer
test_lines = [
    "Artículo 1. Objeto de la Ley",        # esperado alto
    "P99~ct cíe Pilra, o t ( 4 b",          # esperado bajo
    "El presente proyecto de ley regula",  # esperado alto
    "0E7-700/25",                          # esperado bajo
    "Honorable Cámara de Diputados",       # esperado alto
]

print("=== Test OCR Confidence ===")
for line in test_lines:
    conf = _ocr_confidence(line, legal_dict=LEGAL_SPANISH_DICT)
    print(f"Score {conf:.2f}: {line[:50]}")


In [ ]:
import re
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# Fase 0 — Extracción de firmas digitales (llevan a metadatos, no al cuerpo)
# ─────────────────────────────────────────────────────────────────────────────
_SIGNATURE_RE = re.compile(
    r"Digitally\s+signed\s+by\b[^\n]+"
    r"(?:\n(?![ \t]*\n)(?!(?:Art[íi]culo|T[íi]tulo|Cap[íi]tulo)\b)[^\n]{1,120}){0,7}",
    re.IGNORECASE,
)


def extract_signature_blocks(text: str) -> tuple[str, list[str]]:
    signatures: list[str] = []

    def _collect(m: re.Match) -> str:
        signatures.append(m.group(0).strip())
        return ""

    cleaned = _SIGNATURE_RE.sub(_collect, text)
    return re.sub(r"\n{3,}", "\n\n", cleaned).strip(), signatures


# ─────────────────────────────────────────────────────────────────────────────
# Fase 0b — Unión de palabras cortadas por salto de línea
# ─────────────────────────────────────────────────────────────────────────────
def _join_hyphenated_linebreaks(text: str) -> str:
    return re.sub(
        r"([A-Za-záéíóúñÁÉÍÓÚÑ])-\n[ \t]*([A-Za-záéíóúñÁÉÍÓÚÑ])",
        r"\1\2",
        text,
    )


def _normalize_confusable_caps(line: str) -> str:
    """Corrige OCR en tokens casi-mayúscula con dígitos (V1LLARRUEL -> VILLARRUEL)."""
    mapping = str.maketrans({"0": "O", "1": "I", "5": "S"})

    def _fix_token(m: re.Match) -> str:
        token = m.group(0)
        return token.translate(mapping)

    return re.sub(r"\b[\wÁÉÍÓÚÑ]{4,}\b", lambda m: _fix_token(m) if any(ch.isdigit() for ch in m.group(0)) else m.group(0), line)


# ─────────────────────────────────────────────────────────────────────────────
# Helpers de detección y normalización línea a línea
# ─────────────────────────────────────────────────────────────────────────────
def _norm_line(line: str) -> str:
    return re.sub(r"\s+", " ", line).strip().lower()


def _extract_candidate_header_footer_lines(page_text: str, top_n: int = 4, bottom_n: int = 4) -> list[str]:
    lines = [ln.strip() for ln in page_text.splitlines() if ln.strip()]
    if not lines:
        return []
    head = lines[:top_n]
    tail = lines[-bottom_n:] if len(lines) > top_n else []
    return head + tail


def _is_protected_legal_line(line: str) -> bool:
    l = line.strip()
    if not l:
        return False
    if re.match(r"^(art[íi]culo|art\.)\s*\d+", l, re.IGNORECASE):
        return True
    if re.match(r"^(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", l, re.IGNORECASE):
        return True
    return False


def detect_boilerplate_lines_from_pages(docs: list, min_repeat_pages: int = 2) -> set[str]:
    counter: Counter[str] = Counter()
    for d in docs:
        seen_this_page: set[str] = set()
        for ln in _extract_candidate_header_footer_lines(d.page_content):
            if _is_protected_legal_line(ln):
                continue
            n = _norm_line(ln)
            if not n or len(n) < 6:
                continue
            seen_this_page.add(n)
        for n in seen_this_page:
            counter[n] += 1
    return {ln for ln, freq in counter.items() if freq >= min_repeat_pages}


def is_obvious_header_footer_line(line: str) -> bool:
    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return False
    # Expedientes reales y variantes OCR (OD, 0D, 0E7, etc.)
    if re.search(r"\b(?:[A-Z0-9]{1,4})-\d{2,5}/\d{2,4}\b", l, re.IGNORECASE):
        return True
    if len(l) <= 28 and len(re.findall(r"[^\w\s]", l)) >= 4:
        return True
    if re.search(r"aniversario|palacio legislativo|h\.?\s*camara|c[aá]mara|gde", l, re.IGNORECASE):
        return True
    return False


def _is_obvious_stamp_line(line: str) -> bool:
    """Detecta líneas de sello/margen/paginación típicas del OCR roto."""
    l = line.strip()
    if not l or _is_protected_legal_line(l):
        return False
    if re.fullmatch(r"[•·.\-_*~]+", l):
        return True
    if re.fullmatch(r"\d{1,3}", l):
        return True
    # Patrón tipo _4 4- o similares de margen
    if re.fullmatch(r"_?\d+(?:\s+\d+)?-?", l):
        return True
    # Mucho símbolo + dígitos en línea corta
    sym = len(re.findall(r"[~_\\|{}\[\]<>@#$%^&*.,;:()\-]", l))
    dig = len(re.findall(r"\d", l))
    if len(l) < 70 and sym >= 4 and dig >= 2:
        return True
    # Garbled alfanumérico con slashes y puntuación
    if len(l) < 80 and re.search(r"[A-Za-z].*[~/\\].*\d|\d.*[~/\\].*[A-Za-z]", l):
        return True
    # Mixed case + slash/coma/guion (ej: cediAde/i-bef,a)
    if len(l) < 80 and re.search(r"[a-z][A-Z]|[A-Z][a-z][A-Z]", l) and re.search(r"[/,\-_]", l):
        return True
    return False


def _is_ocr_garbage_line(line: str, confidence_threshold: float = 0.6, legal_dict: set = None) -> bool:
    if legal_dict is None:
        legal_dict = LEGAL_SPANISH_DICT
    confidence = _ocr_confidence(line, context_lines=None, legal_dict=legal_dict)
    return confidence < confidence_threshold


def normalize_ocr_line(line: str) -> str:
    # 1) Unicode/control
    line = line.replace("\ufffd", "")
    line = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x80-\x9f]", "", line)
    # 2) escapes OCR
    line = line.replace("\u00ad", "")
    line = line.replace("`", "")
    line = re.sub(r"([A-Za-zÁÉÍÓÚáéíóúñÑ])\\'([A-Za-zÁÉÍÓÚáéíóúñÑ])", r"\1\2", line)  # co\'n -> con
    line = re.sub(r"\\(['\"])", r"\1", line)
    # 3) ordinales
    line = re.sub(r"(?<=\d)\s*[º°o]\b", "°", line)
    line = re.sub(r"(?i)\b(art[íi]culo\s+)([1-9])0\b", r"\1\2°", line)  # Artículo 80 -> 8°
    # 4) basura de puntuación suelta
    line = re.sub(r"([A-Za-záéíóúñ])\)\.", r"\1.", line)  # come). -> come.
    # 5) confusiones alfanuméricas en mayúsculas
    line = _normalize_confusable_caps(line)
    # 6) espacios
    line = re.sub(r"\s+", " ", line).strip()
    return line


def _reflow_preserving_legal_structure(lines: list[str]) -> str:
    out: list[str] = []
    buf: list[str] = []

    def flush_buf() -> None:
        nonlocal buf
        if buf:
            out.append(" ".join(buf).strip())
            buf = []

    for line in lines:
        if not line:
            flush_buf()
            if out and out[-1] != "":
                out.append("")
            continue
        if _is_protected_legal_line(line):
            flush_buf()
            out.append(line)
            continue
        buf.append(line)

    flush_buf()
    text = "\n".join(out)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _dedup_near_duplicate_paragraphs(text: str) -> str:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    kept: list[str] = []
    seen_norms: list[str] = []

    def norm(p: str) -> str:
        p = p.lower()
        p = re.sub(r"\s+", " ", p)
        p = re.sub(r"[^\w\sáéíóúñ]", "", p)
        return p.strip()

    for p in paragraphs:
        n = norm(p)
        if not n:
            continue
        if n in seen_norms:
            continue
        overlapped = False
        for old_n in seen_norms:
            if len(n) > 180 and len(old_n) > 180 and (n in old_n or old_n in n):
                overlapped = True
                break
        if overlapped:
            continue
        kept.append(p)
        seen_norms.append(n)

    return "\n\n".join(kept)


# ─────────────────────────────────────────────────────────────────────────────
# Pipeline principal de limpieza
# ─────────────────────────────────────────────────────────────────────────────
def clean_pdf_text_remove_headers_footers(docs: list, confidence_threshold: float = 0.6) -> tuple[str, list[str], dict]:
    boilerplate = detect_boilerplate_lines_from_pages(docs, min_repeat_pages=max(2, len(docs) // 4))
    cleaned_pages: list[str] = []
    all_signatures: list[str] = []
    seal_lines: list[str] = []
    stats = {
        "lines_processed": 0,
        "lines_boilerplate": 0,
        "lines_header_footer": 0,
        "lines_stamps": 0,
        "lines_ocr_garbage": 0,
        "lines_kept": 0,
    }

    for d in docs:
        page_text = _join_hyphenated_linebreaks(d.page_content)
        page_text, page_sigs = extract_signature_blocks(page_text)
        all_signatures.extend(page_sigs)

        kept_lines: list[str] = []
        for raw in page_text.splitlines():
            line = raw.strip()
            stats["lines_processed"] += 1

            if not line:
                kept_lines.append("")
                continue

            if _is_protected_legal_line(line):
                kept_lines.append(normalize_ocr_line(line))
                stats["lines_kept"] += 1
                continue

            n = _norm_line(line)
            if n in boilerplate:
                stats["lines_boilerplate"] += 1
                seal_lines.append(line)
                continue
            if is_obvious_header_footer_line(line):
                stats["lines_header_footer"] += 1
                seal_lines.append(line)
                continue
            if _is_obvious_stamp_line(line):
                stats["lines_stamps"] += 1
                seal_lines.append(line)
                continue
            if _is_ocr_garbage_line(line, confidence_threshold=confidence_threshold, legal_dict=LEGAL_SPANISH_DICT):
                stats["lines_ocr_garbage"] += 1
                seal_lines.append(line)
                continue

            kept_lines.append(normalize_ocr_line(line))
            stats["lines_kept"] += 1

        page_clean = _reflow_preserving_legal_structure(kept_lines)
        cleaned_pages.append(page_clean)

    cleaned_text = "\n\n".join(cleaned_pages)
    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text).strip()
    cleaned_text = _dedup_near_duplicate_paragraphs(cleaned_text)

    # Guardrail final: limpia islas de ruido entre párrafos legales
    cleaned_text = re.sub(
        r"\n\n(?:[\W\d_~\\/.,;:()\-]{2,}|[A-Za-z0-9~_/.,;:()\-]{1,40})\n\n(?=(?:Art[íi]culo|T[íi]tulo|Cap[íi]tulo)\b)",
        "\n\n",
        cleaned_text,
        flags=re.IGNORECASE,
    )

    stats["seal_lines_detected"] = len(seal_lines)
    stats["seal_lines_sample"] = seal_lines[:10]
    return cleaned_text, all_signatures, stats


# ─────────────────────────────────────────────────────────────────────────────
# Segmentación en unidades legales
# ─────────────────────────────────────────────────────────────────────────────
def split_legal_articles(text: str, extra_metadata: dict | None = None) -> list[dict]:
    lines = text.splitlines()
    sections: list[dict] = []
    current_kind = "preambulo"
    current_title = "PREAMBULO"
    buffer: list[str] = []

    article_re = re.compile(r"^\s*(art[íi]culo|art\.)\s*([\dA-Za-zº°./-]+)", re.IGNORECASE)
    heading_re = re.compile(r"^\s*(t[íi]tulo|cap[íi]tulo|secci[óo]n|libro|anexo)\b", re.IGNORECASE)
    heading_stack: list[str] = []
    doc_meta = extra_metadata or {}

    def flush():
        nonlocal buffer, current_kind, current_title
        content = "\n".join(buffer).strip()
        if content:
            sections.append({
                "kind": current_kind,
                "title": current_title,
                "path": " > ".join(heading_stack),
                "text": content,
                **doc_meta,
            })
        buffer = []

    for raw_line in lines:
        line = normalize_ocr_line(raw_line)
        if not line:
            buffer.append("")
            continue

        article_match = article_re.match(line)
        if article_match:
            flush()
            current_kind = "articulo"
            current_title = line[:160]
            buffer.append(line)
            continue

        heading_match = heading_re.match(line)
        if heading_match:
            flush()
            current_kind = heading_match.group(1).lower()
            current_title = line[:160]
            heading_stack.append(current_title)
            buffer.append(line)
            continue

        buffer.append(line)

    flush()
    return sections


# ─────────────────────────────────────────────────────────────────────────────
# Ejecución del pipeline
# ─────────────────────────────────────────────────────────────────────────────
pdf_text_clean, doc_signatures, cleaning_stats = clean_pdf_text_remove_headers_footers(
    docs,
    confidence_threshold=0.65,
)

# Si viene de flujo proyecto/OCR, garantizamos metadatos mínimos por chunk
if "doc_metadata" not in globals() or not isinstance(doc_metadata, dict):
    doc_metadata = {
        "source": "BORA_OCR",
        "status": "PROYECTO",
        "law_id": None,
        "infoleg_id": None,
        "titulo": "Proyecto (OCR)",
        "search_url": None,
        "ver_norma_url": None,
        "source_url": pdf_para_ocr_path,
        "sha256_hash": None,
    }

article_units = split_legal_articles(pdf_text_clean, extra_metadata=doc_metadata)

print("Chars original:", len(pdf_text))
print("Chars limpio:", len(pdf_text_clean))
print("Firmas extraídas:", len(doc_signatures))
print("Sello/ruido extraído:", cleaning_stats.get("seal_lines_detected", 0))
print("Unidades legales detectadas:", len(article_units))
print("Articulos detectados:", sum(1 for item in article_units if item["kind"] == "articulo"))
print("\n=== Estadísticas de limpieza ===")
for key, val in cleaning_stats.items():
    if key != "seal_lines_sample":
        print(f"  {key}: {val}")

print("\n=== Muestra de sello/ruido extraído ===")
for s in cleaning_stats.get("seal_lines_sample", [])[:5]:
    print(" -", s)


### 9. Inspección del resultado normalizado
Estas celdas muestran el texto limpio y las unidades legales detectadas para revisar manualmente si la normalización conservó el contenido jurídico correcto.

In [ ]:
# Vista rápida del texto limpio y normalizado
pdf_text_clean


In [ ]:
len(article_units)

# Muestra de unidades legales normalizadas
for i, item in enumerate(article_units[:5], start=1):
    print(f"#{i} | {item['kind']} | {item['title']}")
    if item["path"]:
        print("path:", item["path"])
    print(item["text"][:500])
    print("-" * 80)

### 10. Payload de ingesta al backend
Acá se arma el payload que después se puede mandar al endpoint de ingesta para comparar el preprocesamiento del notebook contra el pipeline del backend.

In [ ]:
noise_samples = [
    "cediAde/i-bef,a",
    "P9 9~",
    "P99~ct cíe Pilra",
    "0E7-700/25",
    "0D-700/25",
    "_4 4-",
    "V1LLARRUEL",
    "co\\'n",
    "come).",
]

print("=== Chequeo de patrones de ruido ===")
for s in noise_samples:
    found = s in pdf_text_clean
    print(f"{s!r}: {'ENCONTRADO' if found else 'OK (no aparece)'}")

print("\n=== Chequeos esperados de normalización ===")
print("'VILLARRUEL' presente:", "VILLARRUEL" in pdf_text_clean)
print("'V1LLARRUEL' presente:", "V1LLARRUEL" in pdf_text_clean)
print("'Artículo 8°' presente:", "Artículo 8°" in pdf_text_clean or "ARTÍCULO 8°" in pdf_text_clean)
print("'Artículo 80' presente:", "Artículo 80" in pdf_text_clean or "ARTÍCULO 80" in pdf_text_clean)

print("\n=== Sello/Firma a metadatos ===")
print("Firmas extraídas:", len(doc_signatures))
print("Líneas sello/ruido extraídas:", cleaning_stats.get("seal_lines_detected", 0))


### 9a. Chequeo puntual de ruido reportado
Validación rápida para confirmar si los patrones concretos detectados en ingesta siguen presentes tras la limpieza.

In [ ]:
# 2) Ingesta de documento desde path local (PDF/imagen/texto)
local_pdf_path = "/Users/sandrogarcia/GitRespos/sinmentiras.ar backend/tests/pdfs/104088.pdf"  # <- cambia esto

ingest_payload = {
    'file_path': local_pdf_path,
    # 'minio_path': 'minio://my-bucket/contexto/ley-ejemplo.pdf',  # alternativa
    'presentado_por': 'Dip. Juan Perez',
    'proyecto_tipo': 'modificacion',  # 'base' o 'modificacion'
    'ley_base': 'Ley 27.275',
    'iniciado_en': 'Camara de Diputados',
    'expediente_diputados': '1234-D-2026',
    'expediente_senado': '5678-S-2026',
    'publicado_en': 'Boletin Oficial de la Republica Argentina',
    'fecha': '2026-04-10',
    'ley_numero': '27.804',
    'chunk_size': 700,
    'chunk_overlap': 120,
    'replace_existing': True,
    'metadata': {'fuente': 'manual-notebook', 'tipo': 'normativa'}
}


## Capa de noticias (complementaria a leyes por n8n + MQ)

Este bloque agrega una fuente de noticias para enriquecer el contexto.

- La fuente principal de verdad siguen siendo las leyes que llegan por n8n/MQ.
- Las noticias se indexan por separado y se usan como contexto adicional.
- Luego calculamos un indice de verdad comparando afirmaciones contra el corpus legal.

### 11. Dependencias para capa de noticias
Este primer paso de la capa complementaria instala el parser de RSS que vamos a usar para traer artículos periodísticos.

In [ ]:
%pip install -q feedparser

### 12. Búsqueda de URLs de noticias
Estas celdas consultan Google News RSS y generan una lista deduplicada de links por tema para enriquecer el contexto con prensa.

In [ ]:
import feedparser
from urllib.parse import quote

# Google News RSS por tema (sin API key)
temas = [
    "congreso argentina ley inteligencia artificial",
    "congreso argentina regulacion datos personales",
    "camara diputados tecnologia argentina",
]


def buscar_urls_noticias(temas, max_por_tema=5):
    urls = []
    for tema in temas:
        rss_url = f"https://news.google.com/rss/search?q={quote(tema)}&hl=es-419&gl=AR&ceid=AR:es-419"
        feed = feedparser.parse(rss_url)
        for entry in feed.entries[:max_por_tema]:
            if hasattr(entry, "link"):
                urls.append(entry.link)

    # Deduplicar manteniendo orden
    return list(dict.fromkeys(urls))


news_urls = buscar_urls_noticias(temas, max_por_tema=5)
len(news_urls), news_urls[:3]

### 13. Descarga de contenidos periodísticos
Con la lista de URLs armada, este bloque usa un loader web para bajar el contenido bruto de las noticias seleccionadas.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader_noticias = WebBaseLoader(web_path=news_urls)
docs_noticias_raw = loader_noticias.load()

len(docs_noticias_raw)

### 14. Limpieza del texto de noticias
Acá se remueve ruido típico de sitios periodísticos para quedarnos con texto utilizable antes de crear embeddings.

In [ ]:
from langchain_core.documents import Document
import re


# Limpieza simple para reducir ruido de navegación/publicidad
def limpiar_texto_noticia(texto: str) -> str:
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(
        r"(Leia também|Publicidade|Assine|Compartilhe).*",
        "",
        texto,
        flags=re.IGNORECASE,
    )
    return texto.strip()


docs_noticias = []
for d in docs_noticias_raw:
    texto_limpo = limpiar_texto_noticia(d.page_content)
    if len(texto_limpo) < 400:
        continue

    meta = dict(d.metadata) if d.metadata else {}
    meta["tipo_fonte"] = "noticia"
    docs_noticias.append(Document(page_content=texto_limpo, metadata=meta))

len(docs_noticias)

### 15. Chunking de noticias
Este paso divide las noticias limpias en fragmentos manejables para indexarlas y recuperarlas como contexto complementario.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

news_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
)

news_chunks = news_splitter.split_documents(docs_noticias)
len(news_chunks)

### 16. Vector store y retrieval de noticias
Aquí se construye el índice vectorial de noticias y un retriever simple para probar búsquedas temáticas.

In [ ]:
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

# Embeddings con OpenAI (evita depender de HFEmbeddingsAdapter no definido)
embeddings = OpenAIEmbeddings(
    api_key=openai_api_key,
    model="text-embedding-3-small",
)

vectorstore_noticias = InMemoryVectorStore.from_documents(
    documents=news_chunks,
    embedding=embeddings,
)

retriever_noticias = vectorstore_noticias.as_retriever(search_kwargs={"k": 3})
retriever_noticias.invoke("ley de inteligencia artificial argentina")[:1]

### 17. Evaluación de índice de verdad
Este bloque define el prompt y la función que comparan afirmaciones periodísticas contra el contexto legal recuperado.

In [ ]:
import json
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt_indice_verdad = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Eres un verificador de consistencia.
Debes puntuar de 0 a 1 cuanto coincide una afirmacion periodistica con el contexto legal.
Responde SOLO JSON valido con este formato:
{"indice_verdad": 0.0, "justificacion": "texto breve"}
""",
        ),
        (
            "human",
            "Afirmacion: {afirmacion}\n\nContexto legal:\n{contexto_leyes}",
        ),
    ]
)

cadena_indice_verdad = prompt_indice_verdad | modelo | StrOutputParser()


def calcular_indice_verdad(afirmacion: str, k_leyes: int = 3):
    leyes_rel = retriever.invoke(afirmacion)[:k_leyes]
    contexto_leyes = "\n\n".join([d.page_content for d in leyes_rel])

    raw = cadena_indice_verdad.invoke(
        {"afirmacion": afirmacion, "contexto_leyes": contexto_leyes}
    )

    try:
        resultado = json.loads(raw)
    except Exception:
        resultado = {"indice_verdad": None, "justificacion": raw}

    resultado["afirmacion"] = afirmacion
    return resultado


# Ejemplo de uso
calcular_indice_verdad("El Congreso ya aprobo una ley integral de IA en Argentina")

In [ ]:
# Validación local de URLs de metadata sin pegarle a Infoleg
_test_id = 25552
_test_source_actualizado = "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm"
_test_source_norma = re.sub(r"texact\.htm(?:\?.*)?$", "norma.htm", _test_source_actualizado, flags=re.IGNORECASE)

print("source_norma:", _test_source_norma)
print("source_actualizado:", _test_source_actualizado)
print("vinculos_actualiza_url:", _url_vinculos(_test_id, modo=1))
print("vinculos_actualizado_por_url:", _url_vinculos(_test_id, modo=2))

In [ ]:
import importlib.util
from pathlib import Path
import hashlib
import json

if "texto_fuente" not in globals() or not isinstance(texto_fuente, str) or not texto_fuente.strip():
    raise ValueError("No existe texto_fuente en memoria. Ejecutá antes la celda de carga y chunking de la ley.")

if "split_ley_semantico_articulos" not in globals():
    raise ValueError("No existe split_ley_semantico_articulos en memoria. Ejecutá la celda del splitter del notebook.")

# Carga directa por ruta para evitar conflictos de import en el kernel
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "app" else cwd
rag_service_path = repo_root / "app" / "services" / "rag_service.py"

spec = importlib.util.spec_from_file_location("rag_service_compare", rag_service_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f"No se pudo cargar módulo desde {rag_service_path}")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
LocalRAGService = module.LocalRAGService

# Split notebook (fuente de preparación)
nb_chunks, nb_notes = split_ley_semantico_articulos(texto_fuente)

# Split backend (rag_service)
rag = LocalRAGService()
be_chunks, be_notes = rag._split_ley_semantico_articulos(texto_fuente)

# Normalización para comparación exacta/estable
def _norm_chunks(chunks):
    return [
        {
            "kind": c.get("kind"),
            "title": c.get("title"),
            "text": c.get("text"),
            "path": c.get("path"),
            "articulo_nro": c.get("articulo_nro"),
            "articulo_suffix": c.get("articulo_suffix"),
            "articulo_ref": c.get("articulo_ref"),
            "capitulo": c.get("capitulo"),
        }
        for c in chunks
    ]

def _norm_notes(notes):
    return {str(k): list(v) for k, v in sorted(notes.items(), key=lambda x: str(x[0]))}

nb_chunks_n = _norm_chunks(nb_chunks)
be_chunks_n = _norm_chunks(be_chunks)
nb_notes_n = _norm_notes(nb_notes)
be_notes_n = _norm_notes(be_notes)

same_chunks = nb_chunks_n == be_chunks_n
same_notes = nb_notes_n == be_notes_n

nb_hash = hashlib.sha256(json.dumps(nb_chunks_n, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
be_hash = hashlib.sha256(json.dumps(be_chunks_n, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()

report = {
    "ley": 20744,
    "same_chunks": same_chunks,
    "same_notes": same_notes,
    "nb_chunks": len(nb_chunks_n),
    "be_chunks": len(be_chunks_n),
    "nb_notes_refs": len(nb_notes_n),
    "be_notes_refs": len(be_notes_n),
    "nb_chunks_sha256": nb_hash,
    "be_chunks_sha256": be_hash,
}

print(json.dumps(report, ensure_ascii=False, indent=2))

if not same_chunks:
    max_len = min(len(nb_chunks_n), len(be_chunks_n))
    diff_idx = None
    for i in range(max_len):
        if nb_chunks_n[i] != be_chunks_n[i]:
            diff_idx = i
            break
    if diff_idx is None and len(nb_chunks_n) != len(be_chunks_n):
        diff_idx = max_len
    print("\nPrimer diff de chunks en índice:", diff_idx)
    if diff_idx is not None and diff_idx < len(nb_chunks_n):
        print("Notebook:", json.dumps(nb_chunks_n[diff_idx], ensure_ascii=False, indent=2)[:2000])
    if diff_idx is not None and diff_idx < len(be_chunks_n):
        print("Backend:", json.dumps(be_chunks_n[diff_idx], ensure_ascii=False, indent=2)[:2000])

if not same_notes:
    nb_keys = set(nb_notes_n.keys())
    be_keys = set(be_notes_n.keys())
    print("\nKeys solo notebook:", sorted(nb_keys - be_keys)[:20])
    print("Keys solo backend:", sorted(be_keys - nb_keys)[:20])
    common = sorted(nb_keys & be_keys)
    for k in common:
        if nb_notes_n[k] != be_notes_n[k]:
            print("Primer diff notes en key:", k)
            print("Notebook notes:", json.dumps(nb_notes_n[k], ensure_ascii=False, indent=2)[:2000])
            print("Backend notes:", json.dumps(be_notes_n[k], ensure_ascii=False, indent=2)[:2000])
            break

assert same_chunks and same_notes, "El split del backend no coincide exactamente con el del notebook."

In [ ]:
import os
from pathlib import Path
print('cwd:', Path.cwd())
print('parent:', Path.cwd().parent)
print('exists cwd/app:', (Path.cwd() / 'app').exists())
print('exists parent/app:', (Path.cwd().parent / 'app').exists())